### **Authors:**

Juan Navarro - s1097545

*Add your names + s number when possible*


## Part 1: Loading the data

**Load the train, dev and test sets into pandas DataFrames.**

**What we do:**
1. Read the three CSV files (train, dev and test).
2. Check the shape and columns of each set (snippet ID, 'text', 'author').
3. Fix the text: contractions appear with a double quote instead of an apostrophe (e.g. wasn"t), so we restore the apostrophe (wasn't).
4. Check the number of snippets per author.

**Output:** three cleaned DataFrames: 'train', 'dev' and 'test'.

In [3]:
# Required library
%pip install pandas

Note: you may need to restart the kernel to use updated packages.


In [ ]:
import re
import pandas as pd

#1 Import the csv files as pandas dataframes
train = pd.read_csv('data/pan2627_train_data.csv', index_col=0).rename_axis('id')
dev   = pd.read_csv('data/pan2627_dev_data.csv',   index_col=0).rename_axis('id')
test  = pd.read_csv('data/pan2627_test_data.csv',  index_col=0).rename_axis('id')

#2 Check shapes, columns and author counts
for name, df in [('train', train), ('dev', dev), ('test', test)]:
    print(f'{name}: {df.shape[0]} snippets, columns: {list(df.columns)}')

counts = train['author'].value_counts().sort_index()
print()
print('Number of different authors in train:', train['author'].nunique())
print(f'Fewest snippets for one author: {counts.min()}, most: {counts.max()}')
print()
print('Snippets per author (train):')
print(counts.to_frame('snippets'))

train: 2138 snippets, columns: ['text', 'author']
dev: 268 snippets, columns: ['text', 'author']
test: 267 snippets, columns: ['text', 'author']

Number of different authors in train: 20
Fewest snippets for one author: 51, most: 150

Snippets per author (train):
         snippets
author           
29783          91
240213         98
512464        112
560480        150
583064        130
583994         83
748687        120
806976        130
870118        141
910821         92
967934         99
1112924        93
1220273       100
1276465       121
1497577       109
2750536       132
2855986       117
2943978        51
3439302        98
6234395        71


In [ ]:
#3 Fix the text (i.e. the contractions)
for df in (train, dev, test):
    df['text'] = df['text'].str.replace(r'(?<=[A-Za-z])"(?=[a-z])', "'", regex=True)

# Check that it worked
print(train['text'].iloc[0][:300])

My legs were a bit shaky, so he wrapped an arm around my waist to steady me, "Come on." There was no arguing with the Englishman, and he led me from the room as Italy tended to Germany. The hallways were filled with recovering soldiers, but for the most part the base was quiet. I wasn't used to it, 


## Part 2: Feature extraction

**Goal:** To turn each snippet into numbers that the classifier can use. At least 50 features, both lexical and syntactic organised into groups.

**What we do:**
1. Define the feature groups: lexical, character-level, function words, syntactic (POS tags) and fanfiction-specific.

    - Lexical: average word length, spread of word length, average sentence length, spread of sentence length...
    - Character-level: frequency of ecah punctuation mark, share of uppercase letters, share of digits...
    - Function words: relative frequency of common words (the, of, but...)
    - syntactic (POS tags): relative frequency of each POS tag (noun, verb, adjectiv, adverb...)
    - Fanfiction-specific: share of text inside dialogue quotes, number of dialogue segments, average dialogue length...

2. Write one function per group that takes the snippets and returns a table with one column per feature.
3. Apply the functions to train, dev and test so all three sets have exactly the same columns.
4. Combine the groups into one feature table per set.
5. Check the total number of features (at least 50) and the number of features per group.

**Output:** one feature table for each of 'train', 'dev' and 'test' and a list that shows which features belong to which group.

## Part 3: Classifier and tuning

**Goal:** To build a classifier that predicts the author of a snippet from its features, and tune it using the training and development sets.

**What we do:**
1. Scale the features so they are on a simialr range. The scaler is fitted on train only and then applied to dev and test.
2. Train a few candidate classifiers on the train set:

    - Logistic Regression
    - Linear SVM
    - Random Forest

3. Compare the candidates on the dev set using macro F1.
4. Tune the setings of the best classifiers on the dev set.
5. Keep the best classifier with its settings.

**Output:** the final classifier with its tuned settings and a table with the dev macro F1 of each candidate.

## Part 4: Evaluation

**Goal:** measure how well the final classifier performs on the development set, and to check whether feature selection helps. The target is at least 0.7

**What we do:**
1. Preditc the authors of the dev snippets with the final classifier from Part 3.
2. Calculate the scores:

    - Macro F1 (the main score, because the classes are imbalanced)
    - Accuracy
    - Precision, recall and F1 for each author

3. Plot a confusion matrix to see which authors get mixed up. We need this for the failure analysis in the report.
4. Test feature selection (e.g. keeping only the k best features) and compare the dev macro F1 with and without it.
5. Decide whether feature selection is useful in our case.

**Output:** the dev scores (overall and per author), the confusion matrix and a comparison of the results with and without feature selection.

## Part 5: Ablation analysis

**Goal:** find out which feature group is the most informative for the classifier by removing one group at a time and checking how the performance changes.

**What we do:**
1. Train the final classifier with all feature groups and record the dev macro F1. This is the baseline (no groups removed).
2. For each feature group:

    - Remove all features of that group
    - Retrain the classifier with the same settings
    - Record the dev macro F1

3. Plot the results as a bar chart in the style of Figure 1 of the assignment: one bar for the baseline and one bar for each removed group.
4. Compare the bars: the group dropping the most in F1 score is the most informative.

**Output:** a bar chart saved in the results folder and a table with the dev macro F1 for each removed group.

## Part 6: Test set

**Goal:** run the final classifier on the unseen test set and see how well it generalises. The test set is used once and nothing is changed after seeing the result.

**What we do:**
1. Take the final classifier with its settings and the selected features.
2. Predict the authors of the test snippets.
3. Calculate the same scores as for the dev set:

    - Macro F1 and accuracy
    - Precision, recall and F1 for each author

4. Plot the confusion matrix for the test set.
5. Compare the test scores with the dev scores and note any difference.

**Output:** the test scores (overall and per author), the confusion matrix and a comparison of dev and test performance.